# 1.2 How LLMs Process Text: Tokens and Context

When using an AI model, you may see a model described by its **token limit** or **context window**. These terms also appear in model pricing and in discussions about whether a model can analyze a long document. But what does a token actually represent? And what does it mean for a paper to "fit" in a context window?

In this chapter, we'll look at an example of how text becomes tokens. Then, we'll examine how the context window affects the information an LLM can use and why fitting information into that window does not guarantee that the model will use it correctly. Understanding these concepts will help you plan how to work with scientific papers, data, and tools introduced in later chapters.

## 1.2.1 From Text to Tokens

LLMs process text in units called **tokens**. Tokens often resemble words or punctuation, but they aren't identical to words. Let's compare a simple rule for splitting text with a tokenizer used by an LLM.

In [19]:
import re

sentence = "The TP53 gene encodes p53, a tumor supressor protein that helps regulate the cell cycle."

# Groups consecutive word characters and non-word/space characters (punctuation)
regex_pieces = re.findall(r"\w+|[^\w\s]", sentence)
print("Pieces:", " ".join(f"'{p}'" for p in regex_pieces))
print(f"Count: {len(regex_pieces)}")

Pieces: 'The' 'TP53' 'gene' 'encodes' 'p53' ',' 'a' 'tumor' 'supressor' 'protein' 'that' 'helps' 'regulate' 'the' 'cell' 'cycle' '.'
Count: 17


Our rule divided the sentence into 17 pieces by counting words and punctuation. This can give us a quick sense of text length, but it is only a heuristic for token count. A model's tokenizer uses a vocabularly learned from patterns in text. It may keep a common sequence together, split a word into smaller pieces, or include a precedding space as a part of a token.

Next, we'll use `tiktoken`, a library that implements tokenization for OpenAI models, with a specific token vocabulary to show how models using this encoding would actually divide this sentence.

In [20]:
pip install tiktoken


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")
token_ids = encoding.encode(sentence)
token_pieces = [encoding.decode([token_id]) for token_id in token_ids]

print("Tokens:", " ".join([repr(piece) for piece in token_pieces]))
print(f"Count: {len(token_ids)}")

Tokens: 'The' ' TP' '53' ' gene' ' enc' 'odes' ' p' '53' ',' ' a' ' tumor' ' sup' 'ressor' ' protein' ' that' ' helps' ' regulate' ' the' ' cell' ' cycle' '.'
Count: 21


A tokenizer learns which text sequences are useful to represent together, but its boundaries don't necessarily match the words a person sees when reading; a token may include a space, contain only part of a word, or combine characters differently from our regex rule. Different encodings may also produce different token counts for the same text.

This encoding divided the sentence into 21 tokens, compared to the 17 pieces found by our simple rule. This difference is small for one sentence, but differences in token counts can become consequential when processing long papers or making many requests. 

Token counts matter in two practical ways. They affect how much text fits in a model's **context window**, and model providers commonly use them to measure usage and cost. We'll look at the context window next.

## 1.2.2 Working within the Context Window

A model's **context window** is the amount of information that it can work with during a request, measured in tokens. Context window limits vary by model. The input may include your question, documents you provide, and earlier messages included in the conversation. The model's response also uses part of the available token budget.

For example, imagine asking an LLM to compare findings from several papers. The text of the papers, your instructions, and any relevant conversation history all contribute to the request. If the material exceeds the model's limit, you must isolate only the sections of the papers that you need, trim down your prompt, or start a new chat session. But staying within the limit only means the model can receive the material. It does not guarantee that the model will use every relevant detail correctly. In [*NoLiMa: Long-Context Evaluation Beyond Literal Matching*](https://arxiv.org/abs/2502.05167), researchers found that performance on their information-retreival task declined as input length increased, even for models designed to accept much longer inputs.

A model's context window determines how much material it can receive, while its performance determines how effecively it uses that material. When working with a collection of papers, one approach is to retrieve passages relevant to the question and provide those passages to the model, rahter than supplying every paper in full. This is part of **retrieval-augmented generation (RAG)**, which we will explore in {doc}`Section 2.2 <../chapter_2/2.2.llms_for_biology>`.